# 第12章 小干扰稳定分析 - 两区域系统

> Kundur P. *Power System Stability and Control*, McGraw-Hill, 1994, Chapter 12
> Examples 12.5 - 12.6

## 本章概述

分析 Kundur 经典两区域四机系统的小干扰稳定性，识别：
1. **局部振荡模式** (Local modes): 1-2 Hz，同一区域内发电机之间的振荡
2. **区域间振荡模式** (Inter-area mode): ~0.5-0.6 Hz，两个区域之间的振荡

**系统特征：**
- 4 台发电机 (Area 1: G1, G2; Area 2: G3, G4)，每台 900 MVA
- 11 个节点，双回联络线连接两个区域
- 联络线功率流: Area 1 → Area 2 ≈ 400 MW

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from psa4teaching.data.kundur_two_area import create_kundur_two_area_system
from psa4teaching.network import build_ybus
from psa4teaching.stability import (
    analyze_multi_machine,
    analyze_multi_machine_detailed,
)

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 加载两区域系统
sys = create_kundur_two_area_system()
print(f'Loaded: {len(sys["buses"])} buses, {len(sys["generators"])} gens')

In [ ]:
# 构造缩减导纳矩阵 (发电机节点)
full_ybus = build_ybus(sys['lines'], sys['transformers'])
gen_buses = [1, 2, 3, 4]
gen_idx = [full_ybus.bus_indices[b] for b in gen_buses]
Y_gen = full_ybus.Ybus[np.ix_(gen_idx, gen_idx)]

print(f'Full Ybus: {full_ybus.n_bus}x{full_ybus.n_bus}')
print(f'Gen Ybus: {Y_gen.shape[0]}x{Y_gen.shape[1]}')

# 典型初始功角 (潮流结果近似)
delta_0 = np.radians([15, 10, -5, -10])

---
## Example 12.5: 两区域系统特征值分析

### 经典模型 (2n 阶)

使用经典模型（固定 E'），识别局部和区域间振荡模式。

In [ ]:
# ===== Ex 12.5: 经典模型特征值 =====
result_cl = analyze_multi_machine(
    E_primes=[1.05, 1.03, 1.03, 1.01],
    H_list=[6.5, 6.5, 6.175, 6.175],
    D_list=[0.0, 0.0, 0.0, 0.0],
    delta_0_list=list(delta_0),
    Ybus_reduced=Y_gen, verbose=True
)

# 分类振荡模式
osc_modes = [(ev, f, z) for ev, f, z in zip(
    result_cl.eigenvalues, result_cl.frequencies, result_cl.damping_ratios
) if f > 0.01]

local_modes = [(ev, f, z) for ev, f, z in osc_modes if f > 0.8]
interarea_modes = [(ev, f, z) for ev, f, z in osc_modes if f < 0.8]

print(f'\n局部模式 ({len(local_modes)}):')
for ev, f, z in local_modes:
    print(f'  f={f:.2f} Hz, zeta={z:.4f}')

print(f'\n区域间模式 ({len(interarea_modes)}):')
for ev, f, z in interarea_modes:
    print(f'  f={f:.4f} Hz, zeta={z:.4f}')

In [ ]:
# 特征值分布图
fig, ax = plt.subplots(figsize=(10, 8))
for ev, f, z in osc_modes:
    if f < 0.8:
        ax.plot(ev.real, ev.imag, 'ro', markersize=10,
                label='Inter-area' if f == osc_modes[0][1] else '')
    else:
        ax.plot(ev.real, ev.imag, 'bs', markersize=8,
                label='Local' if f == osc_modes[-1][1] else '')

ax.axvline(x=0, color='k', linewidth=0.5, linestyle='--')
ax.axhline(y=0, color='k', linewidth=0.5)
ax.set_xlabel('Real (1/s)')
ax.set_ylabel('Imag (rad/s)')
ax.set_title('Two-Area System Eigenvalues (Classical Model)')
ax.grid(True, alpha=0.3)

handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys())

plt.tight_layout()
plt.savefig('examples/kundur/two_area_eigenvalues.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Example 12.5 (续): 参与因子分析

参与因子 $p_{ij} = |w_{ij} \\cdot v_{ji}|$ 衡量状态变量 $i$ 对模式 $j$ 的参与程度。

- **区域间模式**: G1, G2 vs G3, G4 的功角以相反方向参与
- **局部模式**: 同区域内发电机的功角以相反方向振荡

In [ ]:
# ===== 参与因子分析 =====
pf = result_cl.participation_factors
n_gen = 4

# 识别区域间模式 (最低频率振荡模式)
osc_indices = [i for i, f in enumerate(result_cl.frequencies) if f > 0.01]
if osc_indices:
    interarea_idx = osc_indices[np.argmin([result_cl.frequencies[i] for i in osc_indices])]
    print(f'区域间模式: mode {interarea_idx+1}, f={result_cl.frequencies[interarea_idx]:.4f} Hz')
    print(f'\n参与因子:')
    print(f'{"State":<15} {"Participation":>12}')
    print('-'*30)
    state_names = [f'delta_{i+1}' for i in range(n_gen)] + [f'omega_{i+1}' for i in range(n_gen)]
    for i in range(2*n_gen):
        p_val = pf[i, interarea_idx]
        bar = '#' * int(p_val * 50)
        print(f'{state_names[i]:<15} {p_val:>12.6f} {bar}')

---
## Example 12.5: 详细模型（磁链衰减 + 励磁系统）

使用详细模型（4 阶，含 Eq' 和简化励磁 KA/Te），观察：
1. 励磁系统引入额外状态（负阻尼风险）
2. 特征值移向右半平面

In [ ]:
# ===== Ex 12.5: 详细模型 (4阶, 简化励磁) =====
result_detail = analyze_multi_machine_detailed(
    E_primes=[1.05, 1.03, 1.03, 1.01],
    H_list=[6.5, 6.5, 6.175, 6.175],
    D_list=[0.0, 0.0, 0.0, 0.0],
    delta_0_list=list(delta_0),
    Ybus_reduced=Y_gen,
    Xd_list=[1.8, 1.8, 1.8, 1.8],
    Xd_prime_list=[0.3, 0.3, 0.3, 0.3],
    Xq_list=[1.7, 1.7, 1.7, 1.7],
    Td0_prime_list=[8.0, 8.0, 8.0, 8.0],
    Ka_list=[200, 200, 200, 200],
    Te_list=[0.5, 0.5, 0.5, 0.5],
    verbose=True
)

print(f'\nState matrix: {result_detail.state_matrix.shape[0]}x{result_detail.state_matrix.shape[1]}')
print(f'Eigenvalues: {len(result_detail.eigenvalues)}')

---
## Example 12.6: PSS 对区域间模式的阻尼

**问题：** 无 PSS 时，区域间模式阻尼不足（甚至为负）。

**方案：** 在关键发电机上加装 PSS，通过 $\\Delta\\omega$ 信号提供附加阻尼。

> 注：完整的 PSS 闭环分析需要在状态矩阵中加入 PSS 动态，
> 这将在后续章节中详细演示。这里用 IEEET1 励磁展示高级励磁系统的影响。

In [ ]:
# ===== Ex 12.6: IEEET1 励磁系统的影响 =====
exciters = sys['exciters']  # 4 个 IEEET1

result_ieeet1 = analyze_multi_machine_detailed(
    E_primes=[1.05, 1.03, 1.03, 1.01],
    H_list=[6.5, 6.5, 6.175, 6.175],
    D_list=[1.0, 1.0, 1.0, 1.0],  # 增加阻尼以保持稳定
    delta_0_list=list(delta_0),
    Ybus_reduced=Y_gen,
    Xd_list=[1.8, 1.8, 1.8, 1.8],
    Xd_prime_list=[0.3, 0.3, 0.3, 0.3],
    Xq_list=[1.7, 1.7, 1.7, 1.7],
    Td0_prime_list=[8.0, 8.0, 8.0, 8.0],
    exciter_params_list=exciters,
    verbose=True
)

# 对比: 经典 vs 详细 vs IEEET1
print(f'\n{"="*50}')
print(f'{"Model":<20} {"States":>8} {"Osc Modes":>12}')
print(f'{"-"*50}')
print(f'{"Classical":<20} {8:>8} {len([f for f in result_cl.frequencies if f>0.01]):>12}')
print(f'{"Detailed (KA/Te)":<20} {16:>8} {len([f for f in result_detail.frequencies if f>0.01]):>12}')
print(f'{"Detailed (IEEET1)":<20} {24:>8} {len([f for f in result_ieeet1.frequencies if f>0.01]):>12}')
print(f'{"="*50}')

---
## 本章小结 (多机小干扰稳定)

1. **两区域系统** 是研究区域间振荡的经典测试系统
2. **区域间模式** (~0.5 Hz) 是最危险的振荡模式：阻尼弱、影响范围广
3. **参与因子** 揭示了各发电机对振荡模式的贡献
4. **励磁系统** 改善暂态响应但可能恶化小干扰阻尼（K5 < 0 效应）
5. **PSS** 是抑制低频振荡的主要手段，通过 $\\Delta\\omega$ → 附加励磁信号 → 正阻尼转矩

> Kundur 参考值:
> - 区域间模式: $f \\approx 0.56$ Hz, $\\zeta \\approx -0.02$ (无 PSS)
> - 局部模式 1 (Area 1): $f \\approx 1.12$ Hz
> - 局部模式 2 (Area 2): $f \\approx 1.16$ Hz